# Classifying Chinese Text Difficulty for L2 Learners Using Fine-Tuned BERT
**STATS 507 Final Project** | Xiaolu Zheng | University of Michigan

## How to Run
1. Open in Google Colab: colab.research.google.com
2. Set runtime: **Runtime → Change runtime type → T4 GPU**
3. **Runtime → Run all**
4. Expected total runtime: ~40 minutes on T4 GPU

## Part 0 — Environment Setup

In [ ]:
!pip install datasets transformers torch scikit-learn pandas matplotlib seaborn -q

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.makedirs('figures', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

## Part 1 — Data Loading and Preprocessing

In [ ]:
# Load and parse dataset
# The dataset stores all fields as interleaved plain-text rows with prefixes:
# english: ... | mandarin: ... | pinyin: ... | hsk: ... | difficulty: ...
print('Loading dataset...')
raw = load_dataset('swaption2009/20k-en-zh-translation-pinyin-hsk')

records, current = [], {}
for example in raw['train']:
    line = example['text'].strip()
    if line.startswith('english:'):
        if 'text' in current and 'label' in current:
            records.append(current)
        current = {}
    elif line.startswith('mandarin:') or line.startswith('chinese:'):
        current['text'] = line.split(':', 1)[1].strip()
    elif line.startswith('hsk:'):
        try:
            lv = int(float(line.split(':', 1)[1].strip()))
            if 1 <= lv <= 4: current['label'] = lv
        except ValueError: pass
if 'text' in current and 'label' in current: records.append(current)

df = pd.DataFrame(records)
print(f'Parsed {len(df):,} valid sentences')

In [ ]:
# Data quality checks
print('Missing values:', df.isnull().sum().to_dict())
counts = df['label'].value_counts().sort_index()
print('Label distribution:')
for lv, n in counts.items(): print(f'  HSK {lv}: {n:,} ({n/len(df)*100:.1f}%)')
df['char_len'] = df['text'].str.len()
avg_len = df.groupby('label')['char_len'].mean()
print('Avg char length:', {f'HSK {k}': f'{v:.2f}' for k,v in avg_len.items()})

In [ ]:
# Figure 1: Data distribution
colors = ['#4E79A7','#F28E2B','#59A14F','#E15759']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar([f'HSK {i}' for i in range(1,5)], [counts[i] for i in range(1,5)], color=colors, edgecolor='white')
axes[0].set_title('Sample Count by HSK Level', fontweight='bold', fontsize=12)
axes[0].set_xlabel('HSK Level'); axes[0].set_ylabel('Number of Sentences')
for i,n in enumerate(counts.values): axes[0].text(i, n+60, str(n), ha='center', fontsize=10)
axes[1].bar([f'HSK {i}' for i in range(1,5)], [avg_len[i] for i in range(1,5)], color=colors, edgecolor='white')
axes[1].set_title('Avg Sentence Length by HSK Level', fontweight='bold', fontsize=12)
axes[1].set_xlabel('HSK Level'); axes[1].set_ylabel('Avg Character Count'); axes[1].set_ylim(12,20)
for i,avg in enumerate(avg_len.values): axes[1].text(i, avg+0.1, f'{avg:.2f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('figures/fig1_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 1 saved.')

In [ ]:
# Train/Val/Test split (70/15/15, stratified)
texts = df['text'].tolist(); labels = df['label'].tolist()
X_tr, X_tmp, y_tr, y_tmp = train_test_split(texts, labels, test_size=0.30, random_state=SEED, stratify=labels)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print(f'Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_te):,}')

## Part 2 — Baseline: TF-IDF + Logistic Regression

In [ ]:
# Character-level TF-IDF (unigrams + bigrams)
# Character-level is preferred for Chinese since words are not space-separated
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(1,2), max_features=10000, sublinear_tf=True)
X_tr_tf = tfidf.fit_transform(X_tr)
X_te_tf = tfidf.transform(X_te)
print(f'TF-IDF matrix shape: {X_tr_tf.shape}')

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced',
                               solver='lbfgs', multi_class='multinomial', random_state=SEED)
lr_model.fit(X_tr_tf, y_tr)
y_pred_base = lr_model.predict(X_te_tf)
base_acc = accuracy_score(y_te, y_pred_base)
base_f1  = f1_score(y_te, y_pred_base, average='macro')
print(f'Baseline | Accuracy: {base_acc:.4f} | Macro F1: {base_f1:.4f}')
print(classification_report(y_te, y_pred_base, target_names=['HSK 1','HSK 2','HSK 3','HSK 4'], digits=4))

## Part 3 — BERT Fine-Tuning

In [ ]:
# Tokenize with bert-base-chinese
print('Loading tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
MAX_LEN = 64
def tokenize(texts):
    return tokenizer(texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
print('Tokenizing...')
enc_tr = tokenize(X_tr); enc_val = tokenize(X_val); enc_te = tokenize(X_te)
print('Tokenization complete.')

In [ ]:
# PyTorch Dataset — labels shifted 1-4 to 0-3
class HSKDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor([l-1 for l in labels], dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k,v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

train_ds = HSKDataset(enc_tr, y_tr); val_ds = HSKDataset(enc_val, y_val); test_ds = HSKDataset(enc_te, y_te)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)}')

In [ ]:
# Load model, optimizer, scheduler
print('Loading bert-base-chinese...')
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=4)
model.to(device)
NUM_EPOCHS = 3; LR = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()
print(f'Model loaded. Steps: {total_steps}')

In [ ]:
# Evaluation helper
def evaluate(model, loader):
    model.eval()
    preds, trues, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device); lbls = batch['labels'].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            total_loss += criterion(logits, lbls).item()
            preds.extend(torch.argmax(logits,1).cpu().numpy())
            trues.extend(lbls.cpu().numpy())
    return total_loss/len(loader), accuracy_score(trues,preds), f1_score(trues,preds,average='macro'), preds, trues

In [ ]:
# Training loop
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_f1, best_path = 0.0, 'best_model.pt'
print(f'Training | Epochs:{NUM_EPOCHS} | LR:{LR} | Batch:16')
print('='*65)
for epoch in range(NUM_EPOCHS):
    model.train()
    ep_loss, ep_preds, ep_labels = 0.0, [], []
    for step, batch in enumerate(train_loader):
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device); lbls = batch['labels'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        ep_loss += loss.item()
        ep_preds.extend(torch.argmax(logits,1).cpu().numpy())
        ep_labels.extend(lbls.cpu().numpy())
        if (step+1)%200==0: print(f'  E{epoch+1} step{step+1}/{len(train_loader)} loss={loss.item():.4f}')
    avg_tr = ep_loss/len(train_loader); tr_acc = accuracy_score(ep_labels, ep_preds)
    avg_va, va_acc, va_f1, _, _ = evaluate(model, val_loader)
    train_losses.append(avg_tr); val_losses.append(avg_va)
    train_accs.append(tr_acc);   val_accs.append(va_acc)
    print(f'Epoch{epoch+1}/{NUM_EPOCHS} train_loss={avg_tr:.4f} val_loss={avg_va:.4f} val_acc={va_acc:.4f} val_f1={va_f1:.4f}')
    if va_f1 > best_f1:
        best_f1 = va_f1; torch.save(model.state_dict(), best_path)
        print(f'  ** checkpoint saved (val_f1={best_f1:.4f}) **')
print('='*65)
print(f'Training complete. Best val macro F1 = {best_f1:.4f}')

## Part 4 — Test Set Evaluation and Figures

In [ ]:
# Load best checkpoint and evaluate on test set
model.load_state_dict(torch.load(best_path, map_location=device))
_, bert_acc, bert_f1, bert_preds, bert_trues = evaluate(model, test_loader)
bert_preds_1 = [p+1 for p in bert_preds]; bert_trues_1 = [l+1 for l in bert_trues]
print(f'BERT | Accuracy: {bert_acc:.4f} | Macro F1: {bert_f1:.4f}')
print(classification_report(bert_trues_1, bert_preds_1, target_names=['HSK 1','HSK 2','HSK 3','HSK 4'], digits=4))

In [ ]:
# Figure 2: Confusion matrices side by side
labels_1_4 = [1,2,3,4]; ticks = ['HSK 1','HSK 2','HSK 3','HSK 4']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yt, yp, title in [(axes[0], y_te, y_pred_base, 'TF-IDF + Logistic Regression'),
                           (axes[1], bert_trues_1, bert_preds_1, 'Fine-Tuned BERT')]:
    cm = confusion_matrix(yt, yp, labels=labels_1_4)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ticks, yticklabels=ticks, ax=ax)
    ax.set_title(title, fontweight='bold', fontsize=12); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('figures/fig2_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 2 saved.')

In [ ]:
# Figure 3: Training curves
ep = range(1, NUM_EPOCHS+1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ep, train_losses,'o-',color='#4E79A7',lw=2,ms=6,label='Train')
axes[0].plot(ep, val_losses,  's--',color='#E15759',lw=2,ms=6,label='Val')
axes[0].set_title('Loss per Epoch',fontweight='bold'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_xticks(list(ep)); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, train_accs,'o-',color='#4E79A7',lw=2,ms=6,label='Train')
axes[1].plot(ep, val_accs,  's--',color='#E15759',lw=2,ms=6,label='Val')
axes[1].set_title('Accuracy per Epoch',fontweight='bold'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_xticks(list(ep)); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig3_training_curves.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 3 saved.')

In [ ]:
# Figure 4: Per-class F1 comparison
base_f1_pc = f1_score(y_te, y_pred_base, labels=labels_1_4, average=None)
bert_f1_pc = f1_score(bert_trues_1, bert_preds_1, labels=labels_1_4, average=None)
x = np.arange(4); w = 0.35
fig, ax = plt.subplots(figsize=(8,5))
b1 = ax.bar(x-w/2, base_f1_pc, w, label='TF-IDF+LR', color='#A0B4C8', edgecolor='white')
b2 = ax.bar(x+w/2, bert_f1_pc, w, label='Fine-Tuned BERT', color='#4E79A7', edgecolor='white')
ax.set_title('Per-Class F1: Baseline vs. BERT', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['HSK 1','HSK 2','HSK 3','HSK 4'])
ax.set_xlabel('HSK Level'); ax.set_ylabel('F1 Score'); ax.set_ylim(0,0.75)
ax.legend(); ax.grid(axis='y',alpha=0.3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig4_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 4 saved.')

In [ ]:
# Final results summary
print('='*62)
print('              FINAL RESULTS SUMMARY')
print('='*62)
print(f'{"Model":<38} {"Accuracy":>10} {"Macro F1":>10}')
print('-'*62)
print(f'{"TF-IDF + Logistic Regression":<38} {base_acc:>10.4f} {base_f1:>10.4f}')
print(f'{"Fine-Tuned BERT (bert-base-chinese)":<38} {bert_acc:>10.4f} {bert_f1:>10.4f}')
print('='*62)
print(f'Improvement: Accuracy +{(bert_acc-base_acc)*100:.2f}pp | Macro F1 +{bert_f1-base_f1:.4f}')
print(f'\n{"Level":<10} {"Baseline F1":>14} {"BERT F1":>10}')
print('-'*36)
for i,lv in enumerate(['HSK 1','HSK 2','HSK 3','HSK 4']):
    print(f'{lv:<10} {base_f1_pc[i]:>14.4f} {bert_f1_pc[i]:>10.4f}')
print('='*62)

## Part 5 — Error Analysis

In [ ]:
# Most confused level pairs
cm_bert = confusion_matrix(bert_trues_1, bert_preds_1, labels=labels_1_4)
errors = sorted([(cm_bert[i][j],i+1,j+1) for i in range(4) for j in range(4) if i!=j], reverse=True)
print('Top 5 misclassification pairs (BERT):')
for cnt,tl,pl in errors[:5]: print(f'  True HSK {tl} → Predicted HSK {pl}: {cnt} cases')
err_idx = [i for i,(p,l) in enumerate(zip(bert_preds_1,bert_trues_1)) if p!=l]
print(f'\nTotal errors: {len(err_idx)}/{len(bert_trues_1)} ({len(err_idx)/len(bert_trues_1)*100:.1f}%)')
print('\nSample errors:')
print('-'*70)
for idx in err_idx[:5]:
    print(f'Sentence: {X_te[idx]}')
    print(f'True HSK {bert_trues_1[idx]} → Predicted HSK {bert_preds_1[idx]}')
    print('-'*70)

## All Done!
Figures in `figures/` — download from left file panel in Colab:
- `fig1_data_distribution.png` — Fig. 1
- `fig2_confusion_matrices.png` — Fig. 2
- `fig3_training_curves.png` — Fig. 3
- `fig4_f1_comparison.png` — Fig. 4